# 01 - XML Exploration & Model Testing

Exploratory notebook for:
1. Understanding LiverTox XML structure
2. Testing dataclass models
3. Prototyping parser and deterministic extraction logic
4. Testing LLM extraction prompts

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import re
import json

from livertox_extraction.models import (
    DrugExtraction, ParsedSections, validate_extraction, extraction_from_dict,
    VALID_DILI_SCORES, VALID_INJURY_PATTERNS, VALID_REGULATORY_STATUSES
)

XML_DIR = Path("../livertox")
xml_files = sorted(XML_DIR.glob("*.xml"))
print(f"Found {len(xml_files)} XML files")
print([f.stem for f in xml_files[:10]], "...")

## 1. Parse a single XML and explore structure

In [ ]:
# Pick a drug with rich content
sample_file = XML_DIR / "Zileuton.xml"
tree = ET.parse(sample_file)
root = tree.getroot()

# List all section IDs and titles
print("=== All sections ===")
for sec in root.iter("sec"):
    sid = sec.get("id", "(no id)")
    title_el = sec.find("title")
    title = title_el.text if title_el is not None else "(no title)"
    print(f"  {sid}: {title}")

In [ ]:
def get_section_text(root, section_id_contains):
    """Extract plain text from a section matching the ID pattern."""
    for sec in root.iter("sec"):
        sid = sec.get("id", "")
        if section_id_contains in sid:
            # Get all text content, stripping tags
            text = ET.tostring(sec, encoding="unicode", method="text")
            # Clean up whitespace
            text = re.sub(r"\s+", " ", text).strip()
            return text
    return None

# Test on key sections
for section in ["Hepatotoxicity", "Background", "Mechanism", "Outcome"]:
    text = get_section_text(root, section)
    if text:
        print(f"\n=== {section} (first 200 chars) ===")
        print(text[:200])
    else:
        print(f"\n=== {section}: NOT FOUND ===")

## 2. Test deterministic extraction patterns

In [ ]:
hepatotox_text = get_section_text(root, "Hepatotoxicity")
print("Full hepatotoxicity section:")
print(hepatotox_text)

In [ ]:
# DILI score regex
dili_pattern = r"Likelihood score:\s*([A-E]\*?|X)"
match = re.search(dili_pattern, hepatotox_text)
if match:
    score = match.group(1)
    print(f"DILI score: {score}")
    print(f"Valid? {score in VALID_DILI_SCORES}")
else:
    print("No DILI score found")

In [ ]:
# Enzyme elevation fraction regex
# Patterns: 'X% of patients', 'in X% of', 'occurred in X%'
enzyme_pattern = r"(\d+\.?\d*)%\s*of\s*(?:patients|recipients)"
matches = re.findall(enzyme_pattern, hepatotox_text, re.IGNORECASE)
print(f"Percentage mentions: {matches}")

# More contextual: look for percentages near ALT/aminotransferase/elevation keywords
context_pattern = r"(?:ALT|aminotransferase|enzyme).*?(\d+\.?\d*)%\s*of\s*(?:patients|recipients)"
match = re.search(context_pattern, hepatotox_text, re.IGNORECASE)
if match:
    pct = float(match.group(1))
    print(f"Enzyme elevation rate: {pct}% = {pct/100:.4f} as fraction")

In [ ]:
# Test DILI score extraction across all available XMLs
dili_pattern = r"Likelihood score:\s*([A-E]\*?|X)"

print("=== DILI scores across all drugs ===")
for xml_file in xml_files:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    hepatotox = get_section_text(root, "Hepatotoxicity")
    if hepatotox:
        match = re.search(dili_pattern, hepatotox)
        score = match.group(1) if match else "NOT FOUND"
    else:
        score = "NO HEPATOTOX SECTION"
    print(f"  {xml_file.stem}: {score}")

## 3. Explore case report Key Points tables

In [ ]:
# Parse case report Key Points tables
sample_file = XML_DIR / "Zileuton.xml"
tree = ET.parse(sample_file)
root = tree.getroot()

for table_wrap in root.iter("table-wrap"):
    table_id = table_wrap.get("id", "")
    print(f"\nTable: {table_id}")
    for tr in table_wrap.iter("tr"):
        th = tr.find("th")
        td = tr.find("td")
        if th is not None and td is not None:
            key = (th.text or "").strip().rstrip(":")
            val_text = ET.tostring(td, encoding="unicode", method="text").strip()
            if key and val_text:
                print(f"  {key}: {val_text}")

## 4. Test dataclass models

In [ ]:
# Create a DrugExtraction from what we've found so far
zileuton = DrugExtraction(
    drug_name="Zileuton",
    dili_likelihood_score="D",
    injury_pattern="hepatocellular",
    fraction_patients_with_enzyme_elevation=0.019,
    peak_alt=33.0,
    r_ratio=25.0,
    is_immune_mediated=False,
    onset_time={"min": 4, "max": 8, "unit": "weeks"},
)

# Validate it
errors = validate_extraction(zileuton)
print(f"Validation errors: {errors}")

# Print as JSON
print("\nAs JSON:")
print(zileuton.to_json())

In [ ]:
# Test creating from a dict (simulates LLM output parsing)
llm_output = {
    "drug_name": "Itraconazole",
    "dili_likelihood_score": "B",
    "injury_pattern": "cholestatic",
    "fraction_patients_with_enzyme_elevation": 0.05,
    "is_immune_mediated": True,
    "risk_factors": [
        {"factor": "Pre-existing liver disease", "supporting_quote": "patients with active liver disease"}
    ],
    "some_extra_key": "this should be ignored",
}

drug = extraction_from_dict(llm_output)
print(f"Drug: {drug.drug_name}, DILI={drug.dili_likelihood_score}")
print(f"Risk factors: {drug.risk_factors}")
print(f"Validation: {validate_extraction(drug)}")

In [ ]:
# Test validation catches bad values
bad_drug = DrugExtraction(
    drug_name="BadDrug",
    dili_likelihood_score="F",              # invalid score
    fraction_patients_with_enzyme_elevation=1.5,  # out of range
    peak_alt=-10.0,                          # negative
)
errors = validate_extraction(bad_drug)
print("Errors caught:")
for e in errors:
    print(f"  - {e}")

## 5. Test parser functions (after parser.py is built)

```python
from livertox_extraction.parser import parse_xml

sections = parse_xml(XML_DIR / "Zileuton.xml")
print(sections)
```

## 6. Test deterministic extraction (after deterministic.py is built)

```python
from livertox_extraction.deterministic import extract_deterministic

result = extract_deterministic(sections)
print(result)
```

## 7. Test LLM extraction (after llm_extractor.py is built)

```python
from livertox_extraction.llm_extractor import extract_with_llm

result = extract_with_llm(sections)
print(result)
```